# Raport: Etap 4 - Fine-tuning modelu DistilBERT

## 1. Cel etapu
Celem tego etapu jest dostrojenie (fine-tuning) pretrenowanego modelu językowego `distilbert-base-uncased` na zbiorze danych dotyczących tweetów o katastrofach. W przeciwieństwie do podejścia zero-shot z poprzedniego etapu, model będzie tu uczony na konkretnych przykładach treningowych, co powinno znacząco zwiększyć dokładność klasyfikacji. Wynikowy model zostanie zapisany i użyty do predykcji w kolejnych etapach.

In [2]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate
import numpy as np

## 2. Przygotowanie danych
Wczytujemy wyczyszczony zbiór danych z etapu EDA. Kolumna `target` zostaje przemianowana na `label`, zgodnie z wymaganiami biblioteki Hugging Face. Następnie konwertujemy dane do formatu `Dataset` i dzielimy je na zbiór treningowy (80%) i testowy (20%).

In [ ]:
df = pd.read_csv('../outputs/train_cleaned.csv')
# Переименовываем колонку target в label (так требует HuggingFace)
df = df.rename(columns={'target': 'label'})
df = df[['clean_text', 'label']].dropna()

# Переводим в формат HuggingFace Dataset и делим на train/test
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

## 3. Tokenizacja
Inicjalizujemy tokenizer modelu `distilbert-base-uncased` i definiujemy funkcję tokenizującą, która przekształca surowy tekst w tensory wejściowe dla modelu. Parametry `padding="max_length"` i `max_length=128` zapewniają jednolity rozmiar wejścia; `truncation=True` skraca dłuższe tweety. Tokenizacja aplikowana jest na całym zbiorze w trybie wsadowym (`batched=True`) dla wydajności.

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["clean_text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

## 4. Inicjalizacja modelu i metryki
Ładujemy model `distilbert-base-uncased` z głowicą klasyfikacyjną (`AutoModelForSequenceClassification`) skonfigurowaną dla 2 klas. Jako metrykę ewaluacyjną wybieramy dokładność (Accuracy) z biblioteki `evaluate`. Funkcja `compute_metrics` oblicza ją na podstawie predykcji i rzeczywistych etykiet po każdej epoce walidacji.

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. Trenowanie modelu (Fine-tuning)
Konfigurujemy parametry trenowania: `learning_rate=2e-5` (typowa wartość dla fine-tuningu BERT), `per_device_train_batch_size=16`, `num_train_epochs=3` — trzy epoki są wystarczające dla modeli transformerowych na zbiorach tej wielkości i minimalizują ryzyko przeuczenia. Trener (`Trainer`) z API Hugging Face obsługuje pętlę treningową, ewaluację i zapis wyników automatycznie.

In [17]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss
500,0.412689
1000,0.301852


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1143, training_loss=0.3415546467297972, metrics={'train_runtime': 241.9813, 'train_samples_per_second': 75.502, 'train_steps_per_second': 4.724, 'total_flos': 605044843361280.0, 'train_loss': 0.3415546467297972, 'epoch': 3.0})

**Analiza + interpretacja:**
Fine-tuning modelu DistilBERT na danych treningowych pozwala modelowi nauczyć się specyfiki języka używanego w tweetach o katastrofach. Trzy epoki trenowania stanowią kompromis między dokładnością a ryzykiem przeuczenia (overfitting). Oczekiwana dokładność po fine-tuningu znacząco przewyższa wynik uzyskany metodą zero-shot, potwierdzając skuteczność podejścia transfer learning w zadaniach klasyfikacji tekstu.

## 6. Zapis modelu
Wytrenowany model i tokenizer zostają zapisane lokalnie do katalogu `../disaster_model`, co umożliwia ich ponowne użycie w kolejnych etapach projektu bez konieczności ponownego trenowania.

In [ ]:
model.save_pretrained("../disaster_model")
tokenizer.save_pretrained("../disaster_model")
print("Model zapisany!")

## 7. Podział pracy / wkład członków grupy

Projekt realizowany w trzyosobowej grupie, z częściowo wspólnym wkładem w poszczególne etapy prac.

- **Edgar Lis** – głównie odpowiedzialny za analizę problemu, dobór hiperparametrów trenowania oraz interpretację wyników ewaluacji; wspierał również weryfikację poprawności podziału danych.
- **Vladyslav Kyzylov** – głównie odpowiedzialny za implementację rozwiązania (kodowanie), konfigurację pipeline'u fine-tuningu oraz integrację z danymi z poprzednich etapów.
- **Maciej Rożek** – głównie odpowiedzialny za opracowanie dokumentacji oraz przygotowanie raportu końcowego; wspierał również organizację projektu i weryfikację wyników.

Wszyscy członkowie zespołu brali udział w konsultacjach dotyczących kierunku projektu oraz końcowej weryfikacji rezultatów.